In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/sushantbhardwaj15@gmail.com/FMCG_Analytics/setup_utils_config/Utilities

## DQ Rules

In [0]:
# ── DQ rules per table ────────────────────────────────────────────────────────
# Each entry: (filter condition, fail reason message)

DQ_RULES = {
    "orders": (
        F.col("order_id").isNotNull() & ((F.col("order_qty").isNotNull())),
        "order_id null or quantity is null",
    ),
}

In [0]:
# ── CUSTOMERS, PRODUCTS, GROSS PRICE (full tables from bronze) ────────────────
good_dfs = {}
bad_dfs ={}
bronze_orders = spark.table(f"{catalog}.{bronze_schema}.orders")
for table_name in ["orders"]:
    print(f"\n DQ checks for {table_name}")
    if spark.catalog.tableExists(f"{catalog}.silver.orders"):
        watermark = (
            spark.table(f"{catalog}.silver.{table_name}")
            .agg(F.max("_ingested_at"))
            .collect()[0][0]
        )
        new_rows = bronze_orders.filter(F.col("_ingested_at") > watermark)
        print(f" Watermark: {watermark} — {new_rows.count()} new rows")
    else:
        new_rows = bronze_orders
        print(f" First run record count— {new_rows.count()} rows")

    condition, fail_reason = DQ_RULES[table_name]
    good, bad = split_good_bad(new_rows, condition, fail_reason)
    save_quarantine(bad, f"{catalog}.bronze.{table_name}",f"{table_name}")

    good = good.withColumn(
        "_silver_loaded_at", F.current_timestamp()
    )
    good_dfs[table_name] = good
    bad_dfs[table_name] =bad


In [0]:
# 2. Clean customer_id → keep numeric, else set to 999999
bronze_orders =  good_dfs["orders"]
df_orders = bronze_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to strin
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

## Join with products

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined)

In [0]:
silver_table = f"{catalog}.silver.orders"


if not spark.catalog.tableExists(silver_table):
    (
        df_joined.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .option("mergeSchema", "true")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    (
        silver_delta.alias("silver")
        .merge(
            df_joined.alias("bronze"),
            """
            silver.order_placement_date = bronze.order_placement_date
            AND silver.order_id = bronze.order_id
            AND silver.product_code = bronze.product_code
            AND silver.customer_id = bronze.customer_id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:

write_audit(
    "silver",
    "orders",
    df_joined.count(),
    "SUCCESS",
    f"quarantined={bad_dfs["orders"].count()}"
)